<a href="https://colab.research.google.com/github/wei001we/financial/blob/main/%E6%8A%95%E8%B3%87%E8%81%96%E7%B6%934_0%E9%99%B3%E5%BD%A5%E7%AB%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL INTEGRATED WHEEL ENGINE
# - Macro regime + portfolio risk + stock risk + wheel options
# - KEEP: macro regime decision layer
# - REMOVE: IE Data / Shiller / CAPE / SOX completely
# - Full runnable version
# - NO AUTO ORDER
# ============================================================

!pip -q install yfinance scipy pandas openpyxl xlrd pandas_datareader scikit-learn

import warnings
warnings.filterwarnings("ignore")

import os
import math
import traceback
import numpy as np
import pandas as pd
import yfinance as yf

from scipy.stats import norm
from pandas_datareader import data as pdr
from IPython.display import display

# ============================================================
# 0) BASE CONFIG
# ============================================================

TODAY = pd.Timestamp.today().normalize()
CONTRACT_MULTIPLIER = 100

# ----------------------------
# ACCOUNT / CAPITAL
# ----------------------------
ACCOUNT_EQUITY = 24838.90
FREE_CASH_USD = 8582.90

MAX_TOTAL_WHEEL_ALLOC_PCT = 0.70
MAX_NEW_CSP_ALLOC_PCT = 0.80
CASH_BUFFER_PCT = 0.20

# ----------------------------
# OPTION UNIVERSE
# ----------------------------
OPTION_UNIVERSE = [
    "AAPL","MSFT","GOOGL","AMZN","META",
    "NVDA","TSM","AMD","MU","INTC","ASML","AMAT","TSLA"
]

# ----------------------------
# STOCK HOLDINGS / COST BASIS
# ----------------------------
STOCK_HOLDINGS = {
    # "AAPL": 100,
}
STOCK_COST_BASIS = {
    # "AAPL": 180.0,
}

POSITIONS_SHARES = STOCK_HOLDINGS.copy()
POSITIONS_COST = STOCK_COST_BASIS.copy()

# ----------------------------
# EXISTING SHORT OPTION POSITIONS
# ----------------------------
SHORT_OPTION_POSITIONS = [
    {
        "ticker": "INTC",
        "type": "PUT",
        "expiry": "2026-04-02",
        "strike": 44,
        "contracts": 1,
        "entry_credit": 1.50,
    },
    {
        "ticker": "INTC",
        "type": "PUT",
        "expiry": "2026-04-02",
        "strike": 40,
        "contracts": 1,
        "entry_credit": 0.50,
    },
    {
        "ticker": "INTC",
        "type": "PUT",
        "expiry": "2026-04-02",
        "strike": 42,
        "contracts": 2,
        "entry_credit": 0.92,
    }
]

# ----------------------------
# BASE WHEEL FILTERS
# ----------------------------
PUT_DTE_MIN = 21
PUT_DTE_MAX = 45
CALL_DTE_MIN = 21
CALL_DTE_MAX = 45

PUT_DELTA_MIN = 0.18
PUT_DELTA_MAX = 0.30
CALL_DELTA_MIN = 0.15
CALL_DELTA_MAX = 0.30

MIN_OPEN_INTEREST = 200
MIN_VOLUME = 5
MAX_BID_ASK_SPREAD_PCT = 0.12
MIN_OPTION_PREMIUM = 0.20
MIN_ANNUALIZED_YIELD = 0.08

TARGET_PROFIT_CLOSE_PCT = 0.50
ROLL_DTE_THRESHOLD = 7
ITM_EXTRINSIC_CLOSE_THRESHOLD = 0.15

ALLOW_CC_BELOW_COST_BASIS = False

# ----------------------------
# ROLL RULES
# ----------------------------
ROLL_PUT_SAME_DELTA_MIN = 0.18
ROLL_PUT_SAME_DELTA_MAX = 0.32
ROLL_CALL_SAME_DELTA_MIN = 0.15
ROLL_CALL_SAME_DELTA_MAX = 0.35

ROLL_OUT_DTE_MIN = 21
ROLL_OUT_DTE_MAX = 60

MIN_ROLL_CREDIT = 0.00
PREFER_STRIKE_IMPROVEMENT = True

# ----------------------------
# ASSIGNMENT SIMULATOR
# ----------------------------
SIM_USE_CC_DTE_MIN = 21
SIM_USE_CC_DTE_MAX = 45
SIM_TARGET_CC_DELTA_MIN = 0.15
SIM_TARGET_CC_DELTA_MAX = 0.30

# ----------------------------
# IVR / SKEW
# ----------------------------
ENABLE_IVR_FILTER = True
IVR_MIN_CSP = 0.40
IVR_MIN_CC = 0.25

ENABLE_SKEW_FILTER = True
MIN_PUT_SKEW_PREMIUM = 0.01
MIN_CALL_SKEW_PREMIUM = -0.02

ENABLE_EVENT_FILTER = False
BLOCK_DTE_IF_EARNINGS_WITHIN = 7

# ----------------------------
# DISPLAY
# ----------------------------
DISPLAY_TOP_CSP = 20
DISPLAY_TOP_CC = 20
DISPLAY_TOP_PLAN = 20

# ============================================================
# 1) MACRO / REGIME / PORTFOLIO CONFIG
# ============================================================

FRED_SERIES = {
    "WALCL": "Fed Balance Sheet",
    "RRPONTSYD": "Reverse Repo",
    "WTREGEN": "TGA",
    "SOFR": "SOFR",
    "BAMLH0A0HYM2": "HY Spread",
    "BAMLC0A0CM": "IG Spread",
    "INDPRO": "Industrial Production",
    "RSAFS": "Retail Sales",
    "DGS2": "2Y",
    "DGS10": "10Y",
    "DFII10": "10Y Real Yield",
}

YF_TICKERS = {
    "^VIX": "VIX",
    "^OVX": "OVX",
    "^GVZ": "GVZ",
    "ES=F": "E-mini S&P 500",
    "NQ=F": "E-mini Nasdaq 100",
    "CL=F": "WTI Crude Oil",
}

MACRO_LOOKBACK = 252
MACRO_START = (TODAY - pd.Timedelta(days=900)).strftime("%Y-%m-%d")
MACRO_END = TODAY.strftime("%Y-%m-%d")

MACRO_WEIGHTS = {
    "Liquidity": 0.18,
    "Credit": 0.22,
    "Volatility": 0.22,
    "Growth": 0.15,
    "Rate": 0.13,
    "Geo": 0.10,
}

MACRO_THRESHOLDS = {
    "VIX": 24.0,
    "Credit": 0.90,
    "Vol": 0.90,
    "MinFlags": 2,
}

DATA_UNSTABLE_MAX_LEV = 0.35
DATA_UNSTABLE_CASH_FLOOR = 0.20

SIGMA_TARGET = 0.18
HAIRCUT = 0.85
DD_WALLS = [
    (-0.20, 0.00),
    (-0.15, 0.35),
    (-0.10, 0.50),
    (-0.07, 0.70),
    (-0.05, 0.85),
]

SEMIS_CORE = {"NVDA","AMD","MU","INTC","AMAT","ASML","TSM"}
MIN_STOCK_LIQ_USD = 15_000_000

RISK_RULES = {
    "USE_STOCK_RISK_FILTER": True,
    "MA_DAYS": 200,
    "ATR_DAYS": 14,
    "TRAIL_LOOKBACK": 120,
    "STOP_LOOKBACK": 120,
    "ADD_LOOKBACK": 120,
    "USE_ATR": True,
    "REQUIRE_ABOVE_MA200_FOR_NEW_CSP": True,
    "MAX_DRAWDOWN_FOR_NEW_CSP": -0.25,
    "MAX_ATR_PCT_FOR_NEW_CSP": 0.08,
    "HIGH_ATR_PENALTY": 0.12,
    "BELOW_MA200_PENALTY": 0.18,
    "DEEP_DD_PENALTY": 0.15,
}

ENABLE_MACRO_GATE = True
ENABLE_PORTFOLIO_RISK_GATE = True
ENABLE_STOCK_RISK_GATE = True

# 保留 macro regime；完全移除 IE/CAPE/SOX overlay
BLOCK_NEW_CSP_IF_CRISIS = True
BLOCK_NEW_CSP_IF_LFINAL_LT = 0.35

REGIME_CSP_SCALAR = {
    "RISK-ON": 1.00,
    "NEUTRAL": 0.85,
    "LATE-CYCLE": 0.70,
    "RISK-OFF": 0.45,
    "DATA-UNSTABLE": 0.30,
    "CRISIS": 0.00,
}

REGIME_DELTA_CAP = {
    "RISK-ON": PUT_DELTA_MAX,
    "NEUTRAL": min(PUT_DELTA_MAX, 0.27),
    "LATE-CYCLE": min(PUT_DELTA_MAX, 0.24),
    "RISK-OFF": min(PUT_DELTA_MAX, 0.18),
    "DATA-UNSTABLE": min(PUT_DELTA_MAX, 0.18),
    "CRISIS": 0.00,
}

REGIME_IVR_MIN = {
    "RISK-ON": max(0.35, IVR_MIN_CSP),
    "NEUTRAL": max(0.40, IVR_MIN_CSP),
    "LATE-CYCLE": 0.45,
    "RISK-OFF": 0.55,
    "DATA-UNSTABLE": 0.55,
    "CRISIS": 999.0,
}

# ============================================================
# 2) GENERIC HELPERS
# ============================================================

def safe_float(x, default=np.nan):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default

def fmt_pct(x):
    return "NA" if pd.isna(x) else f"{x:.2%}"

def fmt_num(x):
    return "NA" if pd.isna(x) else f"{x:,.2f}"

def clamp(x, lo, hi):
    return float(np.clip(x, lo, hi))

def days_to_expiry(expiry_str):
    return int((pd.to_datetime(expiry_str).normalize() - TODAY).days)

def stock_contract_capacity(shares):
    return int(np.floor(float(shares) / 100.0))

def annualize_simple_return(ret, dte):
    if pd.isna(ret) or pd.isna(dte) or dte <= 0:
        return np.nan
    return float(ret * 365.0 / dte)

def safe_display(df, round_cols=None, pct_cols=None, n=4):
    if df is None or df.empty:
        print("No data.")
        return
    x = df.copy()
    round_cols = round_cols or []
    pct_cols = pct_cols or []
    for c in round_cols:
        if c in x.columns:
            x[c] = pd.to_numeric(x[c], errors="coerce").round(n)
    for c in pct_cols:
        if c in x.columns:
            x[c] = pd.to_numeric(x[c], errors="coerce") * 100
            x[c] = x[c].round(2)
    display(x)

def _to_1d_series(x, name=None):
    if x is None:
        return None
    if isinstance(x, pd.DataFrame):
        if x.shape[1] == 1:
            x = x.iloc[:, 0]
        else:
            x = x.iloc[:, -1]
    if not isinstance(x, pd.Series):
        x = pd.Series(x)
    x = x.dropna()
    if name is not None:
        x = x.copy()
        x.name = name
    return x

def force_1d(x):
    if x is None:
        return np.array([], dtype=float)
    if isinstance(x, pd.DataFrame):
        if x.shape[1] == 1:
            x = x.iloc[:, 0]
        else:
            x = x.iloc[:, -1]
    if isinstance(x, pd.Series):
        return np.asarray(x.values).reshape(-1)
    return np.asarray(x).reshape(-1)

def _extract_close_series(df):
    if df is None or df.empty:
        return pd.Series(dtype=float)

    if isinstance(df.columns, pd.MultiIndex):
        if ("Close", "") in df.columns:
            s = df[("Close", "")]
        elif "Close" in df.columns.get_level_values(0):
            s = df["Close"]
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
        else:
            return pd.Series(dtype=float)
    else:
        if "Close" not in df.columns:
            return pd.Series(dtype=float)
        s = df["Close"]

    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]

    return pd.Series(s).dropna().astype(float)

def safe_download_price_history(ticker, period="1y", interval="1d"):
    try:
        df = yf.download(
            ticker,
            period=period,
            interval=interval,
            auto_adjust=True,
            progress=False
        )
        if df is None or df.empty:
            return pd.DataFrame()
        return df
    except Exception:
        return pd.DataFrame()

def safe_last_price(ticker):
    df = safe_download_price_history(ticker, period="20d", interval="1d")
    s = _extract_close_series(df)
    if s.empty:
        return np.nan
    return float(s.iloc[-1])

def safe_fred(code: str, start: str, end: str) -> pd.Series:
    try:
        s = pdr.DataReader(code, "fred", start, end)[code]
        s.name = code
        return s
    except Exception:
        return pd.Series(dtype=float, name=code)

def safe_yf_close(ticker: str, start: str, end: str, interval="1d") -> pd.Series:
    try:
        df = yf.download(ticker, start=start, end=end, interval=interval, progress=False, auto_adjust=True)
        if df is None or df.empty:
            return pd.Series(dtype=float, name=ticker)
        s = _extract_close_series(df)
        s.name = ticker
        return s
    except Exception:
        return pd.Series(dtype=float, name=ticker)

def to_bday_ffill_limit(s: pd.Series, limit_days: int = 10) -> pd.Series:
    if s is None or s.empty:
        return s
    s = s.dropna()
    if s.empty:
        return s

    diffs = pd.Series(s.index).diff().dropna().dt.days
    med_gap = float(diffs.median()) if len(diffs) else 1.0

    if med_gap >= 20:
        lim = 90
    elif med_gap >= 5:
        lim = 25
    else:
        lim = limit_days

    idx = pd.date_range(s.index.min(), s.index.max(), freq="B")
    return s.reindex(idx).ffill(limit=int(lim))

def rolling_z(s: pd.Series, lookback: int, minp: int = 60):
    if s is None or s.empty:
        return s

    mu = s.rolling(lookback, min_periods=minp).mean()
    sd = s.rolling(lookback, min_periods=minp).std()
    z = (s - mu) / sd
    z = z.where(sd > 1e-6)

    if len(z) and pd.isna(z.iloc[-1]):
        last_valid = z.dropna()
        if not last_valid.empty:
            z.iloc[-1] = last_valid.iloc[-1]

    return z.clip(-5, 5)

def pct_chg(s: pd.Series, n: int) -> pd.Series:
    if s is None or s.empty:
        return s
    return s.pct_change(n)

def diff_n(s: pd.Series, n: int) -> pd.Series:
    if s is None or s.empty:
        return s
    return s.diff(n)

# ============================================================
# 3) OPTION PRICING HELPERS
# ============================================================

def bs_put_delta(S, K, T, r, sigma):
    if min(S, K, T, sigma) <= 0:
        return np.nan
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1) - 1.0

def bs_call_delta(S, K, T, r, sigma):
    if min(S, K, T, sigma) <= 0:
        return np.nan
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1)

def option_mid(bid, ask, last):
    bid = np.nan if pd.isna(bid) else float(bid)
    ask = np.nan if pd.isna(ask) else float(ask)
    last = np.nan if pd.isna(last) else float(last)

    if not pd.isna(bid) and not pd.isna(ask) and ask >= bid and ask > 0:
        return (bid + ask) / 2.0
    return last

def spread_pct(bid, ask, mid):
    if pd.isna(bid) or pd.isna(ask) or pd.isna(mid) or mid <= 0:
        return np.nan
    return float((ask - bid) / mid)

# ============================================================
# 4) MACRO REGIME ENGINE
# ============================================================

def classify_regime(latest):
    L = latest.get("Liquidity", np.nan)
    C = latest.get("Credit", np.nan)
    V = latest.get("Volatility", np.nan)
    G = latest.get("Growth", np.nan)
    R = latest.get("Rate", np.nan)
    cov = latest.get("Coverage", 0.0)

    if cov < 0.6:
        return "DATA-UNSTABLE"
    if (C >= 1.5) and (V >= 1.2):
        return "CRISIS"
    if (C >= 0.8) or (V >= 0.9):
        return "RISK-OFF"
    if (R >= 0.8) and (L <= 0.3):
        return "LATE-CYCLE"
    if (L >= 0.4) and (G >= 0.2) and (C <= 0.3) and (V <= 0.3):
        return "RISK-ON"
    return "NEUTRAL"

def decision_limits(regime, crisis_on, latest):
    limits = {"max_gross":1.2, "max_net":0.8, "max_leverage":1.0, "cash_floor":0.10, "notes":[]}

    if regime == "DATA-UNSTABLE":
        limits.update({
            "max_gross":0.6,
            "max_net":0.35,
            "max_leverage":DATA_UNSTABLE_MAX_LEV,
            "cash_floor":max(0.20, DATA_UNSTABLE_CASH_FLOOR),
        })
        limits["notes"].append("資料覆蓋不足：保守模式")
        return limits

    if crisis_on or regime == "CRISIS":
        limits.update({"max_gross":0.35, "max_net":0.20, "max_leverage":0.0, "cash_floor":0.35})
        limits["notes"].append("CRISIS：槓桿=0，現金≥35%")
        return limits

    if regime == "RISK-OFF":
        limits.update({"max_gross":0.90, "max_net":0.55, "max_leverage":0.5, "cash_floor":0.20})
        limits["notes"].append("RISK-OFF：降曝險")
        return limits

    if regime == "LATE-CYCLE":
        limits.update({"max_gross":0.95, "max_net":0.60, "max_leverage":0.7, "cash_floor":0.18})
        limits["notes"].append("LATE-CYCLE：控制高估值曝險")
        return limits

    if regime == "RISK-ON":
        limits.update({"max_gross":1.20, "max_net":0.85, "max_leverage":1.0, "cash_floor":0.10})
        limits["notes"].append("RISK-ON：可進攻")
        return limits

    limits.update({"max_gross":1.0, "max_net":0.65, "max_leverage":0.8, "cash_floor":0.15})
    limits["notes"].append("NEUTRAL：中性控風險")
    return limits

def build_macro_dashboard(start, end, lookback, weights, thresholds, disloc_thr=1.8, ffill_limit=10):
    fred = {c: safe_fred(c, start, end) for c in FRED_SERIES}
    yfs  = {t: safe_yf_close(t, start, end) for t in YF_TICKERS}

    series = []
    for s in fred.values():
        if not s.empty:
            series.append(to_bday_ffill_limit(s, limit_days=ffill_limit))
    for s in yfs.values():
        if not s.empty:
            series.append(to_bday_ffill_limit(s, limit_days=ffill_limit))
    if not series:
        return None

    df = pd.concat(series, axis=1).sort_index()

    if "WALCL" in df.columns:
        df["WALCL_YoY"] = pct_chg(df["WALCL"], 252)
    if "DGS2" in df.columns:
        df["DGS2_1M"] = diff_n(df["DGS2"], 21)

    for t in ["ES=F","NQ=F","CL=F"]:
        if t in df.columns:
            df[f"{t}_ret1"] = df[t].pct_change()

    Z = pd.DataFrame(index=df.index)

    def add_z(raw_col, zname):
        if raw_col in df.columns and df[raw_col].dropna().shape[0] > 80:
            Z[zname] = rolling_z(df[raw_col], lookback, minp=60)
        else:
            Z[zname] = np.nan

    add_z("WALCL_YoY", "z_FedBS_YoY")
    add_z("RRPONTSYD", "z_RRP")
    add_z("WTREGEN", "z_TGA")
    add_z("SOFR", "z_SOFR")
    add_z("BAMLH0A0HYM2", "z_HY")
    add_z("BAMLC0A0CM", "z_IG")
    add_z("^VIX", "z_VIX")
    add_z("^OVX", "z_OVX")
    add_z("^GVZ", "z_GVZ")
    add_z("INDPRO", "z_INDPRO")
    add_z("RSAFS", "z_RSAFS")
    add_z("DFII10", "z_Real10")
    add_z("DGS2_1M", "z_2Y_1M")

    for t in ["ES=F","NQ=F","CL=F"]:
        c = f"{t}_ret1"
        if c in df.columns and df[c].dropna().shape[0] > 80:
            Z[f"z_{t}_shock"] = rolling_z(df[c].abs(), lookback, minp=60)
        else:
            Z[f"z_{t}_shock"] = np.nan

    Liquidity = (+1.0*Z["z_FedBS_YoY"]) + (-1.0*Z["z_RRP"]) + (-1.0*Z["z_TGA"]) + (-0.5*Z["z_SOFR"])
    Credit    = (1.2*Z["z_HY"]) + (0.8*Z["z_IG"])
    Volatility= (1.0*Z["z_VIX"].fillna(0)) + (0.4*Z["z_OVX"].fillna(0)) + (0.3*Z["z_GVZ"].fillna(0))
    Geo       = (0.6*Z["z_OVX"].fillna(0)) + (0.6*Z["z_GVZ"].fillna(0)) + (0.2*Z["z_VIX"].fillna(0))
    Growth    = (1.0*Z["z_INDPRO"]) + (0.8*Z["z_RSAFS"])
    Rate      = (1.0*Z["z_Real10"]) + (0.8*Z["z_2Y_1M"])

    Dislocation = (
        0.45 * Z["z_ES=F_shock"].fillna(0) +
        0.35 * Z["z_NQ=F_shock"].fillna(0) +
        0.20 * Z["z_CL=F_shock"].fillna(0)
    )

    factors = pd.DataFrame({
        "Liquidity": Liquidity,
        "Credit": Credit,
        "Volatility": Volatility,
        "Growth": Growth,
        "Rate": Rate,
        "Geo": Geo,
        "Dislocation": Dislocation,
    }, index=df.index)

    cols = ["Liquidity","Credit","Volatility","Growth","Rate","Geo"]
    W = pd.Series(weights)

    avail = factors[cols].notna().astype(float)
    w_eff = avail.mul(W[cols], axis=1)
    w_sum = w_eff.sum(axis=1).replace(0, np.nan)
    w_norm = w_eff.div(w_sum, axis=0)

    factors["TotalScore"] = (factors[cols] * w_norm).sum(axis=1, skipna=True)
    factors["Coverage"] = avail.sum(axis=1) / len(cols)

    latest_row = factors.dropna(how="all").iloc[-1]
    latest = latest_row.to_dict()

    regime = classify_regime(latest)

    vix_latest = df["^VIX"].dropna().iloc[-1] if "^VIX" in df.columns and df["^VIX"].dropna().shape[0] else np.nan
    flags = {
        "VIX > threshold": (not pd.isna(vix_latest)) and (vix_latest > thresholds["VIX"]),
        "CreditStress > threshold": (not pd.isna(latest.get("Credit", np.nan))) and (latest["Credit"] > thresholds["Credit"]),
        "VolScore > threshold": (not pd.isna(latest.get("Volatility", np.nan))) and (latest["Volatility"] > thresholds["Vol"]),
        "Dislocation > thr": (not pd.isna(latest.get("Dislocation", np.nan))) and (latest["Dislocation"] > disloc_thr),
    }

    crisis_on = False
    if regime != "DATA-UNSTABLE":
        base = (sum(bool(v) for k, v in flags.items() if k != "Dislocation > thr") >= thresholds["MinFlags"])
        veto = bool(flags["Dislocation > thr"])
        crisis_on = base or veto

    limits = decision_limits(regime, crisis_on, latest)

    return {
        "raw": df,
        "Z": Z,
        "factors": factors,
        "latest": latest,
        "regime": regime,
        "crisis_flags": flags,
        "crisis_on": crisis_on,
        "limits": limits,
        "disloc_thr": disloc_thr,
        "vix_latest": float(vix_latest) if not pd.isna(vix_latest) else np.nan
    }

# ============================================================
# 5) STOCK SIGNALS / PORTFOLIO RISK
# ============================================================

def fetch_prices_for_pool(tickers, end, days=420):
    px = {}
    for t in tickers:
        s = safe_yf_close(t, start=(pd.to_datetime(end) - pd.Timedelta(days=days*2)).strftime("%Y-%m-%d"), end=end)
        if s is None or s.empty:
            continue
        px[t] = s
    if not px:
        return pd.DataFrame()
    df = pd.concat(px.values(), axis=1)
    df.columns = list(px.keys())
    return df.sort_index().dropna(how="all")

def compute_liquidity_usd(ticker, end, days=60):
    try:
        df = yf.download(ticker, period=f"{days}d", interval="1d", auto_adjust=True, progress=False)
        if df is None or df.empty or "Volume" not in df.columns:
            return np.nan
        close = _extract_close_series(df)
        vol = df["Volume"]
        if isinstance(vol, pd.DataFrame):
            vol = vol.iloc[:, 0]
        v = (close * pd.Series(vol).reindex(close.index)).dropna()
        if v.empty:
            return np.nan
        return float(v.tail(min(20, len(v))).mean())
    except Exception:
        return np.nan

def compute_stock_signals(px_df: pd.DataFrame):
    if px_df is None or px_df.empty:
        return pd.DataFrame()

    out = []
    for t in px_df.columns:
        s = px_df[t].dropna()
        if len(s) < 180:
            continue

        r_6m = float(s.iloc[-1] / s.iloc[-126] - 1) if len(s) >= 126 else np.nan
        r_12m = float(s.iloc[-1] / s.iloc[-252] - 1) if len(s) >= 252 else np.nan
        ma200 = float(s.rolling(200).mean().iloc[-1]) if len(s) >= 200 else np.nan
        trend = float((s.iloc[-1] / ma200 - 1)) if (not pd.isna(ma200) and ma200 > 0) else np.nan
        ret = s.pct_change().dropna()
        vol20 = float(ret.tail(min(20, len(ret))).std(ddof=1) * np.sqrt(252)) if len(ret) >= 20 else np.nan

        out.append({"ticker": t, "mom6": r_6m, "mom12": r_12m, "trend": trend, "vol": vol20})

    if not out:
        return pd.DataFrame()
    return pd.DataFrame(out).set_index("ticker")

def build_target_weights(sig: pd.DataFrame, end, top_n_total=30, semis_min=10):
    if sig is None or sig.empty:
        return pd.Series(dtype=float)

    liq = {t: compute_liquidity_usd(t, end=end) for t in sig.index}
    sig = sig.copy()
    sig["liq_usd"] = pd.Series(liq)
    sig = sig[(sig["liq_usd"].fillna(0) >= MIN_STOCK_LIQ_USD) | (sig.index.isin(SEMIS_CORE))]

    if sig.empty:
        return pd.Series(dtype=float)

    sig["score"] = (
        0.55 * sig["mom12"].fillna(0) +
        0.35 * sig["mom6"].fillna(0) +
        0.20 * sig["trend"].fillna(0) -
        0.10 * sig["vol"].fillna(sig["vol"].median())
    )

    ranked = sig.sort_values("score", ascending=False)
    semis = ranked.loc[ranked.index.intersection(SEMIS_CORE)].copy()
    nonsemis = ranked.loc[~ranked.index.isin(SEMIS_CORE)].copy()

    keep_semis = list(semis.head(semis_min).index) if len(semis) > 0 else []
    remain = top_n_total - len(keep_semis)
    keep_rest = list(nonsemis.head(max(0, remain)).index)
    chosen = list(dict.fromkeys(keep_semis + keep_rest))

    if not chosen:
        return pd.Series(dtype=float)

    sub = sig.loc[chosen].copy()
    v = sub["vol"].replace(0, np.nan).fillna(sub["vol"].median())
    inv = 1.0 / v
    w = inv / inv.sum()
    w = w.clip(upper=0.10)
    w = w / w.sum()
    return w.sort_values(ascending=False)

def compute_holdings_table(positions_shares: dict, end, positions_cost: dict=None):
    positions_cost = positions_cost or {}
    tickers = [t for t, sh in positions_shares.items() if sh and sh != 0]

    if not tickers:
        cols = ["ticker","shares","price","market_value","weight","avg_cost","cost_value","unreal_pnl","unreal_pnl_pct"]
        return pd.DataFrame(columns=cols).set_index("ticker"), 0.0

    prices = {}
    for t in tickers:
        s = safe_yf_close(t, start=(pd.to_datetime(end)-pd.Timedelta(days=15)).strftime("%Y-%m-%d"), end=end)
        prices[t] = float(s.iloc[-1]) if (s is not None and not s.empty) else np.nan

    rows = []
    for t in tickers:
        sh = float(positions_shares[t])
        px = prices.get(t, np.nan)
        mv = sh * px if not pd.isna(px) else np.nan

        avg_cost = positions_cost.get(t, None)
        avg_cost = float(avg_cost) if avg_cost is not None and not (isinstance(avg_cost, float) and np.isnan(avg_cost)) else np.nan
        cost_value = sh * avg_cost if not pd.isna(avg_cost) else np.nan

        unreal = (mv - cost_value) if (not pd.isna(mv) and not pd.isna(cost_value)) else np.nan
        unreal_pct = (unreal / cost_value) if (not pd.isna(unreal) and not pd.isna(cost_value) and abs(cost_value) > 1e-12) else np.nan

        rows.append({
            "ticker": t,
            "shares": sh,
            "price": px,
            "market_value": mv,
            "avg_cost": avg_cost,
            "cost_value": cost_value,
            "unreal_pnl": unreal,
            "unreal_pnl_pct": unreal_pct
        })

    dfh = pd.DataFrame(rows).set_index("ticker")
    total = float(dfh["market_value"].sum(skipna=True))
    dfh["weight"] = (dfh["market_value"] / total) if total > 0 else np.nan
    return dfh.sort_values("market_value", ascending=False), total

def compute_portfolio_returns_from_positions(positions_shares: dict, end, lookback_days=260):
    dfh, total = compute_holdings_table(positions_shares, end, positions_cost=POSITIONS_COST)
    if total <= 0 or dfh.empty:
        raise ValueError("No stock positions")

    tickers = dfh.index.tolist()
    closes = []
    for t in tickers:
        raw = yf.download(t, period=f"{lookback_days}d", interval="1d", auto_adjust=True, progress=False)
        if raw is None or raw.empty or "Close" not in raw.columns:
            continue
        s = _to_1d_series(_extract_close_series(raw), name=t)
        if s is None or s.empty:
            continue
        closes.append(s)

    if not closes:
        raise ValueError("Unable to download position prices")

    px = pd.concat(closes, axis=1).dropna(how="any")
    kept = px.columns.tolist()
    w2 = np.array([dfh.loc[t, "weight"] for t in kept], dtype=float)
    w2 = w2 / w2.sum()
    rets = px.pct_change().dropna()
    port = rets.dot(w2)
    port.name = "port_ret"
    return port

def ann_vol(daily_ret: pd.Series, lookback=60):
    s = daily_ret.dropna()
    if len(s) < 30:
        raise ValueError("Portfolio return history too short")
    w = s.iloc[-min(len(s), lookback):]
    return float(w.std(ddof=1) * np.sqrt(252))

def equity_curve(daily_ret: pd.Series, base=1.0):
    return base * (1.0 + daily_ret).cumprod()

def drawdown_now(daily_ret: pd.Series):
    eq = equity_curve(daily_ret)
    peak = eq.cummax()
    dd = (eq - peak) / peak
    return float(dd.iloc[-1])

def estimate_beta(port_ret: pd.Series, bench="^NDX", lookback_days=220):
    raw = yf.download(bench, period=f"{lookback_days}d", interval="1d", auto_adjust=True, progress=False)
    close = _extract_close_series(raw) if isinstance(raw, pd.DataFrame) else None
    close = _to_1d_series(close, name="close")
    if close is None or close.empty:
        raise ValueError("benchmark close unavailable")
    b = close.pct_change().dropna().copy()
    b.name = "b"
    p = _to_1d_series(port_ret, name="p")
    dfb = pd.concat([p, b], axis=1).dropna()
    if len(dfb) < 60:
        raise ValueError(f"beta sample too small: {len(dfb)}")
    cov = float(np.cov(dfb["p"].values, dfb["b"].values, ddof=1)[0, 1])
    var = float(np.var(dfb["b"].values, ddof=1))
    if var <= 1e-12:
        raise ValueError("benchmark variance is zero")
    beta = cov / var
    return float(np.clip(beta, 0.2, 3.0))

def dd_governor(L_raw, dd_now):
    L = float(L_raw)
    for wall, cap in DD_WALLS:
        if dd_now < wall:
            L = min(L, cap)
            break
    return float(max(L, 0.0))

def leverage_from_margin(equity_usd, free_cash_usd, haircut=0.85):
    if equity_usd <= 0:
        return 1.0
    return float(1.0 + (max(0.0, free_cash_usd) * haircut) / equity_usd)

def leverage_engine(sigma_ann, dd_now, regime, crisis_on, limits, equity_usd, free_cash_usd):
    L_vol = (SIGMA_TARGET / sigma_ann) if (sigma_ann and sigma_ann > 1e-9) else 0.0
    L_vol = max(0.0, float(L_vol))
    L_dd = dd_governor(L_vol, dd_now)

    if crisis_on or regime == "CRISIS":
        L_reg = 0.0
    elif regime == "DATA-UNSTABLE":
        L_reg = min(L_dd, float(limits.get("max_leverage", DATA_UNSTABLE_MAX_LEV)))
    else:
        L_reg = min(L_dd, float(limits.get("max_leverage", 1.0)))

    L_margin = leverage_from_margin(equity_usd, free_cash_usd, haircut=HAIRCUT)
    L_final = min(L_reg, L_margin)

    return {
        "L_vol": float(L_vol),
        "L_dd": float(L_dd),
        "L_reg": float(L_reg),
        "L_margin": float(L_margin),
        "L_final": float(max(L_final, 0.0))
    }

def compute_risk_indicators(tickers, end, rules, positions_cost=None):
    positions_cost = positions_cost or {}
    if not tickers:
        return pd.DataFrame()

    end_dt = pd.to_datetime(end)
    start_dt = end_dt - pd.Timedelta(days=max(420, rules.get("MA_DAYS", 200) * 2))
    start = start_dt.strftime("%Y-%m-%d")
    end_s = end_dt.strftime("%Y-%m-%d")

    df = yf.download(
        tickers,
        start=start,
        end=end_s,
        interval="1d",
        auto_adjust=True,
        progress=False,
        group_by="ticker",
        threads=True
    )

    rows = []
    ma_n = int(rules.get("MA_DAYS", 200))
    atr_n = int(rules.get("ATR_DAYS", 14))
    lb = int(max(rules.get("TRAIL_LOOKBACK", 120), rules.get("STOP_LOOKBACK", 120), rules.get("ADD_LOOKBACK", 120)))

    def _one(t, dft):
        if dft is None or dft.empty or ("Close" not in dft.columns):
            return None

        close = dft["Close"].dropna()
        if close.empty:
            return None

        last_close = float(close.iloc[-1])
        ma200 = close.rolling(ma_n).mean()
        last_ma = float(ma200.iloc[-1]) if len(ma200.dropna()) else np.nan

        roll_max = close.rolling(lb, min_periods=min(60, lb)).max()
        last_peak = float(roll_max.iloc[-1]) if len(roll_max.dropna()) else np.nan
        dd = (last_close / last_peak - 1.0) if (not pd.isna(last_peak) and last_peak > 0) else np.nan

        atr_pct = np.nan
        if rules.get("USE_ATR", True) and all(c in dft.columns for c in ["High", "Low", "Close"]):
            high = dft["High"].astype(float)
            low  = dft["Low"].astype(float)
            prev_close = dft["Close"].astype(float).shift(1)
            tr = pd.concat([(high-low).abs(), (high-prev_close).abs(), (low-prev_close).abs()], axis=1).max(axis=1)
            atr = tr.rolling(atr_n, min_periods=min(atr_n, 10)).mean()
            last_atr = float(atr.iloc[-1]) if len(atr.dropna()) else np.nan
            atr_pct = (last_atr / last_close) if (not pd.isna(last_atr) and last_close > 0) else np.nan

        avg_cost = positions_cost.get(t, np.nan)
        try:
            avg_cost = float(avg_cost) if avg_cost is not None else np.nan
        except Exception:
            avg_cost = np.nan

        pnl_pct = (last_close / avg_cost - 1.0) if (not pd.isna(avg_cost) and avg_cost > 0) else np.nan

        return {
            "ticker": t,
            "close": last_close,
            "ma200": last_ma,
            "dd": float(dd) if not pd.isna(dd) else np.nan,
            "atr_pct": float(atr_pct) if not pd.isna(atr_pct) else np.nan,
            "pnl_pct": float(pnl_pct) if not pd.isna(pnl_pct) else np.nan
        }

    if isinstance(df.columns, pd.MultiIndex):
        for t in tickers:
            if t not in df.columns.get_level_values(0):
                continue
            dft = df[t].dropna(how="all")
            r = _one(t, dft)
            if r:
                rows.append(r)
    else:
        t = tickers[0]
        r = _one(t, df)
        if r:
            rows.append(r)

    return pd.DataFrame(rows).set_index("ticker") if rows else pd.DataFrame()

# ============================================================
# 6) BUILD MARKET CONTEXT
# ============================================================

def build_integrated_market_context(
    start=None,
    end=None,
    lookback=252,
    weights=None,
    thresholds=None
):
    start = start or MACRO_START
    end = end or MACRO_END
    weights = weights or MACRO_WEIGHTS
    thresholds = thresholds or MACRO_THRESHOLDS

    out = {
        "macro": None,
        "regime": "NEUTRAL",
        "crisis_on": False,
        "limits": {"max_gross":1.0, "max_net":0.65, "max_leverage":0.8, "cash_floor":0.15, "notes":[]},
        "stock_signals": pd.DataFrame(),
        "target_weights": pd.Series(dtype=float),
        "risk_indicators": pd.DataFrame(),
        "portfolio_risk": None,
        "L_final": 1.0,
        "regime_scalar": 1.0,
        "regime_delta_cap": PUT_DELTA_MAX,
        "regime_ivr_min": IVR_MIN_CSP,
    }

    try:
        macro = build_macro_dashboard(
            start=start,
            end=end,
            lookback=lookback,
            weights=weights,
            thresholds=thresholds,
            disloc_thr=1.8,
            ffill_limit=10
        )
        out["macro"] = macro
        if macro is not None:
            out["regime"] = macro.get("regime", "NEUTRAL")
            out["crisis_on"] = bool(macro.get("crisis_on", False))
            out["limits"] = macro.get("limits", out["limits"])
    except Exception as e:
        print("macro dashboard failed:", e)

    try:
        px_pool = fetch_prices_for_pool(OPTION_UNIVERSE, end=end, days=420)
        sig = compute_stock_signals(px_pool)
        out["stock_signals"] = sig
        if sig is not None and not sig.empty:
            out["target_weights"] = build_target_weights(
                sig,
                end=end,
                top_n_total=min(30, len(sig)),
                semis_min=min(5, max(1, len(sig.index.intersection(SEMIS_CORE))))
            )
    except Exception as e:
        print("stock signals failed:", e)

    try:
        risk_ind = compute_risk_indicators(
            OPTION_UNIVERSE,
            end=end,
            rules=RISK_RULES,
            positions_cost=POSITIONS_COST
        )
        out["risk_indicators"] = risk_ind
    except Exception as e:
        print("risk indicators failed:", e)

    try:
        if POSITIONS_SHARES and sum(float(v) for v in POSITIONS_SHARES.values()) > 0:
            port_ret = compute_portfolio_returns_from_positions(POSITIONS_SHARES, end=end, lookback_days=260)
            sigma_ann = ann_vol(port_ret, lookback=60)
            dd_now = drawdown_now(port_ret)
            beta = estimate_beta(port_ret, bench="^NDX", lookback_days=220)

            lev = leverage_engine(
                sigma_ann=sigma_ann,
                dd_now=dd_now,
                regime=out["regime"],
                crisis_on=out["crisis_on"],
                limits=out["limits"],
                equity_usd=ACCOUNT_EQUITY,
                free_cash_usd=FREE_CASH_USD
            )
            out["portfolio_risk"] = {
                "sigma_ann": sigma_ann,
                "dd_now": dd_now,
                "beta": beta,
                **lev
            }
            out["L_final"] = float(lev["L_final"])
    except Exception as e:
        print("portfolio risk skipped:", e)

    out["regime_scalar"] = REGIME_CSP_SCALAR.get(out["regime"], 0.75)
    out["regime_delta_cap"] = REGIME_DELTA_CAP.get(out["regime"], PUT_DELTA_MAX)
    out["regime_ivr_min"] = REGIME_IVR_MIN.get(out["regime"], IVR_MIN_CSP)

    if out["crisis_on"] or out["regime"] == "CRISIS":
        out["L_final"] = 0.0
    elif out["regime"] == "DATA-UNSTABLE":
        out["L_final"] = min(out["L_final"], DATA_UNSTABLE_MAX_LEV)
    elif out["regime"] == "RISK-OFF":
        out["L_final"] = min(
            out["L_final"],
            safe_float(out["limits"].get("max_leverage", 0.5), 0.5)
        )

    return out

MARKET_CONTEXT = build_integrated_market_context(
    start=MACRO_START,
    end=MACRO_END,
    lookback=MACRO_LOOKBACK,
    weights=MACRO_WEIGHTS,
    thresholds=MACRO_THRESHOLDS
)

def get_market_context():
    global MARKET_CONTEXT
    return MARKET_CONTEXT

def refresh_market_context():
    global MARKET_CONTEXT
    MARKET_CONTEXT = build_integrated_market_context(
        start=MACRO_START,
        end=MACRO_END,
        lookback=MACRO_LOOKBACK,
        weights=MACRO_WEIGHTS,
        thresholds=MACRO_THRESHOLDS
    )
    return MARKET_CONTEXT

# ============================================================
# 7) DECISION-LAYER HOOKS
# ============================================================

def macro_allows_new_csp(mc=None):
    mc = mc or get_market_context()
    regime = mc.get("regime", "NEUTRAL")
    crisis_on = bool(mc.get("crisis_on", False))
    L_final = safe_float(mc.get("L_final", 1.0), 1.0)

    if not ENABLE_MACRO_GATE:
        return True, "macro_gate_disabled"

    if BLOCK_NEW_CSP_IF_CRISIS and (crisis_on or regime == "CRISIS"):
        return False, f"blocked_by_regime={regime}"

    if L_final < BLOCK_NEW_CSP_IF_LFINAL_LT:
        return False, f"blocked_by_L_final={L_final:.2f}"

    return True, "allowed"

def get_ticker_risk_row(ticker, mc=None):
    mc = mc or get_market_context()
    ri = mc.get("risk_indicators", pd.DataFrame())
    if ri is None or ri.empty or ticker not in ri.index:
        return None
    return ri.loc[ticker]

def stock_is_allowed_for_new_csp(ticker, mc=None):
    if not ENABLE_STOCK_RISK_GATE:
        return True, []

    mc = mc or get_market_context()
    row = get_ticker_risk_row(ticker, mc)
    if row is None:
        return True, []

    reasons = []
    close = safe_float(row.get("close", np.nan))
    ma200 = safe_float(row.get("ma200", np.nan))
    dd = safe_float(row.get("dd", np.nan))
    atr_pct = safe_float(row.get("atr_pct", np.nan))

    if RISK_RULES.get("REQUIRE_ABOVE_MA200_FOR_NEW_CSP", True):
        if not pd.isna(close) and not pd.isna(ma200) and close < ma200:
            reasons.append("Below_MA200")

    max_dd = safe_float(RISK_RULES.get("MAX_DRAWDOWN_FOR_NEW_CSP", -0.25), -0.25)
    if not pd.isna(dd) and dd < max_dd:
        reasons.append("Deep_Drawdown")

    max_atr = safe_float(RISK_RULES.get("MAX_ATR_PCT_FOR_NEW_CSP", 0.08), 0.08)
    if not pd.isna(atr_pct) and atr_pct > max_atr:
        reasons.append("ATR_Too_High")

    return len(reasons) == 0, reasons

def stock_risk_score_penalty(ticker, mc=None):
    if not ENABLE_STOCK_RISK_GATE:
        return 0.0

    mc = mc or get_market_context()
    row = get_ticker_risk_row(ticker, mc)
    if row is None:
        return 0.0

    penalty = 0.0
    close = safe_float(row.get("close", np.nan))
    ma200 = safe_float(row.get("ma200", np.nan))
    dd = safe_float(row.get("dd", np.nan))
    atr_pct = safe_float(row.get("atr_pct", np.nan))

    if not pd.isna(close) and not pd.isna(ma200) and close < ma200:
        penalty += safe_float(RISK_RULES.get("BELOW_MA200_PENALTY", 0.18), 0.18)

    if not pd.isna(dd) and dd < safe_float(RISK_RULES.get("MAX_DRAWDOWN_FOR_NEW_CSP", -0.25), -0.25):
        penalty += safe_float(RISK_RULES.get("DEEP_DD_PENALTY", 0.15), 0.15)

    if not pd.isna(atr_pct) and atr_pct > safe_float(RISK_RULES.get("MAX_ATR_PCT_FOR_NEW_CSP", 0.08), 0.08):
        penalty += safe_float(RISK_RULES.get("HIGH_ATR_PENALTY", 0.12), 0.12)

    return float(penalty)

# 完全移除 IE/CAPE/SOX 後，這裡固定 neutral
def get_semi_heat_overlay(ticker, mc=None):
    return {"hot": False, "delta_cap": PUT_DELTA_MAX, "score_penalty": 0.0}

def effective_put_delta_cap(ticker, mc=None):
    mc = mc or get_market_context()
    reg_cap = safe_float(mc.get("regime_delta_cap", PUT_DELTA_MAX), PUT_DELTA_MAX)
    return reg_cap

def effective_ivr_min(mc=None):
    mc = mc or get_market_context()
    return max(IVR_MIN_CSP, safe_float(mc.get("regime_ivr_min", IVR_MIN_CSP), IVR_MIN_CSP))

def csp_contract_scalar_for_ticker(ticker, mc=None):
    mc = mc or get_market_context()
    regime_scalar = safe_float(mc.get("regime_scalar", 1.0), 1.0)
    L_final = safe_float(mc.get("L_final", 1.0), 1.0)
    leverage_scalar = min(1.0, max(0.0, L_final))

    stock_scalar = 1.0
    row = get_ticker_risk_row(ticker, mc)
    if row is not None:
        atr_pct = safe_float(row.get("atr_pct", np.nan))
        if not pd.isna(atr_pct):
            if atr_pct > 0.08:
                stock_scalar *= 0.40
            elif atr_pct > 0.06:
                stock_scalar *= 0.60
            elif atr_pct > 0.04:
                stock_scalar *= 0.80

    final_scalar = regime_scalar * leverage_scalar * stock_scalar
    return float(max(0.0, min(1.0, final_scalar)))

# ============================================================
# 8) OPTION CHAIN
# ============================================================

def get_option_expirations(ticker):
    try:
        tk = yf.Ticker(ticker)
        return list(tk.options) if tk.options else []
    except Exception:
        return []

def get_option_chain(ticker, expiry):
    try:
        tk = yf.Ticker(ticker)
        chain = tk.option_chain(expiry)
        return chain.calls.copy(), chain.puts.copy()
    except Exception:
        return pd.DataFrame(), pd.DataFrame()

def normalize_option_chain(df, ticker, expiry, opt_type, spot, risk_free=0.045):
    if df is None or df.empty:
        return pd.DataFrame()

    x = df.copy()
    x["ticker"] = ticker
    x["expiry"] = expiry
    x["type"] = opt_type
    x["dte"] = days_to_expiry(expiry)

    x["mid"] = x.apply(
        lambda r: option_mid(r.get("bid", np.nan), r.get("ask", np.nan), r.get("lastPrice", np.nan)),
        axis=1
    )
    x["spread_pct"] = x.apply(
        lambda r: spread_pct(r.get("bid", np.nan), r.get("ask", np.nan), r.get("mid", np.nan)),
        axis=1
    )

    if "impliedVolatility" in x.columns:
        x["iv"] = pd.to_numeric(x["impliedVolatility"], errors="coerce")
    else:
        x["iv"] = np.nan

    T = x["dte"] / 365.0

    if opt_type == "PUT":
        x["delta_bs"] = x.apply(
            lambda r: bs_put_delta(
                spot,
                float(r["strike"]),
                float(max(T.loc[r.name], 1 / 365)),
                risk_free,
                float(r["iv"])
            ) if (not pd.isna(r["iv"]) and r["iv"] > 0 and spot > 0 and float(r["strike"]) > 0) else np.nan,
            axis=1
        )
        x["intrinsic"] = np.maximum(x["strike"] - spot, 0.0)
        x["otm_pct"] = (spot - x["strike"]) / spot
    else:
        x["delta_bs"] = x.apply(
            lambda r: bs_call_delta(
                spot,
                float(r["strike"]),
                float(max(T.loc[r.name], 1 / 365)),
                risk_free,
                float(r["iv"])
            ) if (not pd.isna(r["iv"]) and r["iv"] > 0 and spot > 0 and float(r["strike"]) > 0) else np.nan,
            axis=1
        )
        x["intrinsic"] = np.maximum(spot - x["strike"], 0.0)
        x["otm_pct"] = (x["strike"] - spot) / spot

    x["extrinsic"] = x["mid"] - x["intrinsic"]
    x["credit_per_contract"] = x["mid"] * CONTRACT_MULTIPLIER
    x["notional_if_assigned"] = x["strike"] * CONTRACT_MULTIPLIER
    x["annualized_yield"] = np.where(
        x["dte"] > 0,
        (x["credit_per_contract"] / x["notional_if_assigned"]) * (365.0 / x["dte"]),
        np.nan
    )

    cols = [
        "ticker","expiry","type","dte","contractSymbol","strike",
        "bid","ask","lastPrice","mid","volume","openInterest",
        "iv","delta_bs","spread_pct","intrinsic","extrinsic",
        "otm_pct","credit_per_contract","notional_if_assigned","annualized_yield"
    ]
    for c in cols:
        if c not in x.columns:
            x[c] = np.nan
    return x[cols].copy()

# ============================================================
# 9) IV / SKEW
# ============================================================

def approx_iv_rank_proxy(ticker):
    spot = safe_last_price(ticker)
    if pd.isna(spot) or spot <= 0:
        return np.nan, np.nan

    exps = get_option_expirations(ticker)
    if not exps:
        return np.nan, np.nan

    valid = [e for e in exps if PUT_DTE_MIN <= days_to_expiry(e) <= PUT_DTE_MAX]
    if not valid:
        valid = exps[:1]

    expiry = valid[0]
    _, puts = get_option_chain(ticker, expiry)
    if puts.empty or "impliedVolatility" not in puts.columns:
        return np.nan, np.nan

    puts = puts.copy()
    puts["dist"] = (puts["strike"] - spot).abs()
    atm = puts.sort_values("dist").head(1)
    if atm.empty:
        return np.nan, np.nan

    current_iv = safe_float(atm["impliedVolatility"].iloc[0])

    hist = safe_download_price_history(ticker, period="1y", interval="1d")
    close = _extract_close_series(hist)
    if close.empty or len(close) < 60:
        return current_iv, np.nan

    rv20 = close.pct_change().rolling(20).std() * np.sqrt(252)
    rv20 = rv20.dropna()
    if rv20.empty:
        return current_iv, np.nan

    iv_rank_proxy = float((rv20 < current_iv).mean()) if not pd.isna(current_iv) else np.nan
    return current_iv, iv_rank_proxy

def get_atm_contract_iv(df, spot):
    if df is None or df.empty or "impliedVolatility" not in df.columns:
        return np.nan
    x = df.copy()
    x["dist"] = (pd.to_numeric(x["strike"], errors="coerce") - float(spot)).abs()
    atm = x.sort_values("dist").head(1)
    if atm.empty:
        return np.nan
    return safe_float(atm["impliedVolatility"].iloc[0])

def get_put_skew_metrics(ticker, expiry, spot):
    _, puts = get_option_chain(ticker, expiry)
    if puts.empty:
        return {"atm_iv": np.nan, "put_25d_iv": np.nan, "put_skew": np.nan}

    norm_puts = normalize_option_chain(puts, ticker, expiry, "PUT", spot)
    if norm_puts.empty:
        return {"atm_iv": np.nan, "put_25d_iv": np.nan, "put_skew": np.nan}

    atm_iv = get_atm_contract_iv(puts, spot)
    target = norm_puts[(norm_puts["delta_bs"].abs() >= 0.20) & (norm_puts["delta_bs"].abs() <= 0.30)].copy()
    if target.empty:
        return {"atm_iv": atm_iv, "put_25d_iv": np.nan, "put_skew": np.nan}

    target["delta_dist"] = (target["delta_bs"].abs() - 0.25).abs()
    best = target.sort_values(["delta_dist", "strike"]).head(1)
    put_25d_iv = safe_float(best["iv"].iloc[0]) if not best.empty else np.nan
    put_skew = put_25d_iv - atm_iv if (not pd.isna(put_25d_iv) and not pd.isna(atm_iv)) else np.nan
    return {"atm_iv": atm_iv, "put_25d_iv": put_25d_iv, "put_skew": put_skew}

def get_call_skew_metrics(ticker, expiry, spot):
    calls, _ = get_option_chain(ticker, expiry)
    if calls.empty:
        return {"atm_iv": np.nan, "call_25d_iv": np.nan, "call_skew": np.nan}

    norm_calls = normalize_option_chain(calls, ticker, expiry, "CALL", spot)
    if norm_calls.empty:
        return {"atm_iv": np.nan, "call_25d_iv": np.nan, "call_skew": np.nan}

    atm_iv = get_atm_contract_iv(calls, spot)
    target = norm_calls[(norm_calls["delta_bs"] >= 0.20) & (norm_calls["delta_bs"] <= 0.30)].copy()
    if target.empty:
        return {"atm_iv": atm_iv, "call_25d_iv": np.nan, "call_skew": np.nan}

    target["delta_dist"] = (target["delta_bs"] - 0.25).abs()
    best = target.sort_values(["delta_dist", "strike"]).head(1)
    call_25d_iv = safe_float(best["iv"].iloc[0]) if not best.empty else np.nan
    call_skew = call_25d_iv - atm_iv if (not pd.isna(call_25d_iv) and not pd.isna(atm_iv)) else np.nan
    return {"atm_iv": atm_iv, "call_25d_iv": call_25d_iv, "call_skew": call_skew}

def get_earnings_date(ticker):
    if not ENABLE_EVENT_FILTER:
        return pd.NaT
    try:
        tk = yf.Ticker(ticker)
        cal = tk.calendar
        if cal is None or len(cal) == 0:
            return pd.NaT
        if isinstance(cal, pd.DataFrame):
            for idx in cal.index:
                if str(idx).lower().startswith("earnings"):
                    val = cal.loc[idx].iloc[0]
                    return pd.to_datetime(val)
        return pd.NaT
    except Exception:
        return pd.NaT

def earnings_blocked(ticker):
    ed = get_earnings_date(ticker)
    if pd.isna(ed):
        return False, pd.NaT
    days = (pd.to_datetime(ed).normalize() - TODAY).days
    return 0 <= days <= BLOCK_DTE_IF_EARNINGS_WITHIN, ed

# ============================================================
# 10) CAPITAL SNAPSHOT
# ============================================================

def current_stock_holdings_table():
    rows = []
    for t, sh in STOCK_HOLDINGS.items():
        px = safe_last_price(t)
        cb = STOCK_COST_BASIS.get(t, np.nan)
        mv = sh * px if not pd.isna(px) else np.nan
        pnl_pct = (px / cb - 1.0) if (not pd.isna(px) and not pd.isna(cb) and cb > 0) else np.nan
        rows.append({
            "ticker": t,
            "shares": float(sh),
            "price": px,
            "market_value": mv,
            "cost_basis": cb,
            "unreal_pnl_pct": pnl_pct,
            "cc_capacity": stock_contract_capacity(sh)
        })

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index("ticker").sort_values("market_value", ascending=False)

def summarize_short_option_positions():
    if not SHORT_OPTION_POSITIONS:
        return pd.DataFrame(columns=["ticker","type","expiry","dte","strike","contracts","entry_credit"])
    df = pd.DataFrame(SHORT_OPTION_POSITIONS).copy()
    df["dte"] = df["expiry"].map(days_to_expiry)
    return df.sort_values(["ticker","type","expiry"]).reset_index(drop=True)

def total_stock_market_value():
    df = current_stock_holdings_table()
    return float(df["market_value"].sum(skipna=True)) if not df.empty else 0.0

def estimated_total_csp_obligation():
    df = summarize_short_option_positions()
    if df.empty:
        return 0.0
    puts = df[df["type"] == "PUT"].copy()
    if puts.empty:
        return 0.0
    puts["cash_req"] = puts["strike"].astype(float) * CONTRACT_MULTIPLIER * puts["contracts"].astype(int)
    return float(puts["cash_req"].sum())

def wheel_capital_snapshot_integrated(mc=None):
    mc = mc or get_market_context()

    stock_mv = total_stock_market_value()
    short_put_cash = estimated_total_csp_obligation()
    total_deployed = stock_mv + short_put_cash

    limits = mc.get("limits", {})
    max_total_deploy_pct = min(
        MAX_TOTAL_WHEEL_ALLOC_PCT,
        safe_float(limits.get("max_gross", MAX_TOTAL_WHEEL_ALLOC_PCT), MAX_TOTAL_WHEEL_ALLOC_PCT)
    )
    min_cash_floor_pct = max(
        CASH_BUFFER_PCT,
        safe_float(limits.get("cash_floor", CASH_BUFFER_PCT), CASH_BUFFER_PCT)
    )

    max_total_deploy = ACCOUNT_EQUITY * max_total_deploy_pct
    min_cash_buffer = ACCOUNT_EQUITY * min_cash_floor_pct

    return {
        "stock_mv": stock_mv,
        "short_put_cash_obligation": short_put_cash,
        "total_deployed": total_deployed,
        "max_total_deploy": max_total_deploy,
        "available_deploy_room": max(0.0, max_total_deploy - total_deployed),
        "cash_buffer_required": min_cash_buffer,
        "free_cash_now": FREE_CASH_USD,
        "macro_regime": mc.get("regime", "NEUTRAL"),
        "macro_crisis_on": bool(mc.get("crisis_on", False)),
        "L_final": safe_float(mc.get("L_final", 1.0), 1.0),
    }

def max_new_contracts_for_put_integrated(strike, ticker, mc=None):
    mc = mc or get_market_context()
    snap = wheel_capital_snapshot_integrated(mc)

    if strike <= 0:
        return 0

    per_contract_cash = strike * CONTRACT_MULTIPLIER
    usable_cash = max(0.0, FREE_CASH_USD - snap["cash_buffer_required"])
    deploy_room = max(0.0, snap["available_deploy_room"])
    per_ticker_cap = ACCOUNT_EQUITY * MAX_NEW_CSP_ALLOC_PCT
    max_cash_for_name = min(usable_cash, deploy_room, per_ticker_cap)

    base_contracts = max(0, int(np.floor(max_cash_for_name / per_contract_cash)))
    scalar = csp_contract_scalar_for_ticker(ticker, mc)
    final_contracts = int(np.floor(base_contracts * scalar))

    if base_contracts >= 1 and final_contracts == 0 and scalar >= 0.50:
        final_contracts = 1

    return max(0, final_contracts)

# ============================================================
# 11) CSP / CC SCANNERS
# ============================================================

def build_new_csp_recommendations_integrated(mc=None):
    mc = mc or get_market_context()

    macro_ok, macro_reason = macro_allows_new_csp(mc)
    if not macro_ok:
        return pd.DataFrame()

    rows = []

    for ticker in OPTION_UNIVERSE:
        stock_ok, stock_reasons = stock_is_allowed_for_new_csp(ticker, mc)
        if not stock_ok:
            continue

        spot = safe_last_price(ticker)
        if pd.isna(spot) or spot <= 0:
            continue

        exps = get_option_expirations(ticker)
        if not exps:
            continue

        ticker_rows = []
        earnings_flag, earnings_date = earnings_blocked(ticker)

        for e in exps:
            dte = days_to_expiry(e)
            if dte < PUT_DTE_MIN or dte > PUT_DTE_MAX:
                continue

            _, puts = get_option_chain(ticker, e)
            if puts.empty:
                continue

            norm_puts = normalize_option_chain(puts, ticker, e, "PUT", spot)
            if norm_puts.empty:
                continue

            delta_cap = effective_put_delta_cap(ticker, mc)
            ivr_min = effective_ivr_min(mc)

            norm_puts = norm_puts[
                (norm_puts["openInterest"].fillna(0) >= MIN_OPEN_INTEREST) &
                (norm_puts["volume"].fillna(0) >= MIN_VOLUME) &
                (norm_puts["mid"].fillna(0) >= MIN_OPTION_PREMIUM) &
                (norm_puts["spread_pct"].fillna(999) <= MAX_BID_ASK_SPREAD_PCT) &
                (norm_puts["delta_bs"].abs() >= PUT_DELTA_MIN) &
                (norm_puts["delta_bs"].abs() <= delta_cap) &
                (norm_puts["annualized_yield"].fillna(0) >= MIN_ANNUALIZED_YIELD)
            ]

            if norm_puts.empty:
                continue

            current_iv, iv_rank_proxy = approx_iv_rank_proxy(ticker)
            skew = get_put_skew_metrics(ticker, e, spot)

            norm_puts["spot"] = spot
            norm_puts["iv_current_proxy"] = current_iv
            norm_puts["iv_rank_proxy"] = iv_rank_proxy
            norm_puts["atm_iv"] = skew["atm_iv"]
            norm_puts["put_25d_iv"] = skew["put_25d_iv"]
            norm_puts["put_skew"] = skew["put_skew"]
            norm_puts["earnings_blocked"] = earnings_flag
            norm_puts["earnings_date"] = earnings_date

            row = get_ticker_risk_row(ticker, mc)
            if row is not None:
                for c in ["close","ma200","dd","atr_pct","pnl_pct"]:
                    norm_puts[c] = safe_float(row.get(c, np.nan))
            else:
                norm_puts["close"] = np.nan
                norm_puts["ma200"] = np.nan
                norm_puts["dd"] = np.nan
                norm_puts["atr_pct"] = np.nan
                norm_puts["pnl_pct"] = np.nan

            norm_puts["macro_regime"] = mc.get("regime", "NA")
            norm_puts["macro_crisis_on"] = bool(mc.get("crisis_on", False))
            norm_puts["macro_L_final"] = safe_float(mc.get("L_final", np.nan))
            norm_puts["macro_gate_reason"] = macro_reason
            norm_puts["stock_filter_reasons"] = ",".join(stock_reasons) if stock_reasons else ""
            norm_puts["semi_hot"] = False

            if ENABLE_IVR_FILTER:
                norm_puts = norm_puts[norm_puts["iv_rank_proxy"].fillna(-1) >= ivr_min]

            if ENABLE_SKEW_FILTER:
                norm_puts = norm_puts[norm_puts["put_skew"].fillna(-999) >= MIN_PUT_SKEW_PREMIUM]

            if ENABLE_EVENT_FILTER and earnings_flag:
                norm_puts = norm_puts.iloc[0:0]

            if norm_puts.empty:
                continue

            base_score = (
                0.25 * norm_puts["annualized_yield"].fillna(0) +
                0.15 * norm_puts["openInterest"].fillna(0).rank(pct=True) +
                0.10 * norm_puts["volume"].fillna(0).rank(pct=True) +
                0.10 * norm_puts["otm_pct"].fillna(0) +
                0.20 * norm_puts["iv_rank_proxy"].fillna(0) +
                0.15 * norm_puts["put_skew"].fillna(0) -
                0.05 * norm_puts["spread_pct"].fillna(0.5)
            )

            penalty = stock_risk_score_penalty(ticker, mc)
            if mc.get("regime") in ["RISK-OFF", "LATE-CYCLE", "DATA-UNSTABLE"]:
                penalty += 0.05

            norm_puts["score"] = base_score - penalty
            norm_puts["trade_framework"] = np.where(
                norm_puts["put_skew"].fillna(-999) >= MIN_PUT_SKEW_PREMIUM,
                "SELL_PUT_ON_RICH_DOWNSIDE_SKEW",
                "SELL_PUT_STANDARD"
            )

            ticker_rows.append(norm_puts)

        if not ticker_rows:
            continue

        csp = pd.concat(ticker_rows, axis=0, ignore_index=True).sort_values(
            ["score", "annualized_yield"], ascending=[False, False]
        ).reset_index(drop=True)

        if csp.empty:
            continue

        best = csp.iloc[0].copy()
        max_contracts = max_new_contracts_for_put_integrated(float(best["strike"]), ticker=ticker, mc=mc)
        if max_contracts <= 0:
            continue

        best["recommended_contracts"] = max_contracts
        best["cash_required"] = float(best["strike"]) * CONTRACT_MULTIPLIER * max_contracts
        best["expected_credit_total"] = float(best["credit_per_contract"]) * max_contracts
        rows.append(best)

    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows)
    return out.sort_values(["score", "annualized_yield"], ascending=[False, False]).reset_index(drop=True)

def scan_covered_calls(ticker, shares_held, cost_basis=None):
    eligible_contracts = stock_contract_capacity(shares_held)
    if eligible_contracts <= 0:
        return pd.DataFrame()

    spot = safe_last_price(ticker)
    if pd.isna(spot) or spot <= 0:
        return pd.DataFrame()

    exps = get_option_expirations(ticker)
    if not exps:
        return pd.DataFrame()

    earnings_flag, earnings_date = earnings_blocked(ticker)
    rows = []

    for e in exps:
        dte = days_to_expiry(e)
        if dte < CALL_DTE_MIN or dte > CALL_DTE_MAX:
            continue

        calls, _ = get_option_chain(ticker, e)
        if calls.empty:
            continue

        norm_calls = normalize_option_chain(calls, ticker, e, "CALL", spot)
        if norm_calls.empty:
            continue

        norm_calls = norm_calls[
            (norm_calls["openInterest"].fillna(0) >= MIN_OPEN_INTEREST) &
            (norm_calls["volume"].fillna(0) >= MIN_VOLUME) &
            (norm_calls["mid"].fillna(0) >= MIN_OPTION_PREMIUM) &
            (norm_calls["spread_pct"].fillna(999) <= MAX_BID_ASK_SPREAD_PCT) &
            (norm_calls["delta_bs"].fillna(0) >= CALL_DELTA_MIN) &
            (norm_calls["delta_bs"].fillna(0) <= CALL_DELTA_MAX)
        ]

        if not ALLOW_CC_BELOW_COST_BASIS and cost_basis is not None and not pd.isna(cost_basis):
            norm_calls = norm_calls[norm_calls["strike"] >= float(cost_basis)]

        if norm_calls.empty:
            continue

        current_iv, iv_rank_proxy = approx_iv_rank_proxy(ticker)
        skew = get_call_skew_metrics(ticker, e, spot)

        norm_calls["spot"] = spot
        norm_calls["eligible_contracts"] = eligible_contracts
        norm_calls["cost_basis"] = cost_basis
        norm_calls["iv_current_proxy"] = current_iv
        norm_calls["iv_rank_proxy"] = iv_rank_proxy
        norm_calls["atm_iv"] = skew["atm_iv"]
        norm_calls["call_25d_iv"] = skew["call_25d_iv"]
        norm_calls["call_skew"] = skew["call_skew"]
        norm_calls["earnings_blocked"] = earnings_flag
        norm_calls["earnings_date"] = earnings_date

        if ENABLE_IVR_FILTER:
            norm_calls = norm_calls[norm_calls["iv_rank_proxy"].fillna(-1) >= IVR_MIN_CC]

        if ENABLE_SKEW_FILTER:
            norm_calls = norm_calls[norm_calls["call_skew"].fillna(-999) >= MIN_CALL_SKEW_PREMIUM]

        if ENABLE_EVENT_FILTER and earnings_flag:
            norm_calls = norm_calls.iloc[0:0]

        if norm_calls.empty:
            continue

        rows.append(norm_calls)

    if not rows:
        return pd.DataFrame()

    out = pd.concat(rows, axis=0, ignore_index=True)
    out["score"] = (
        0.25 * out["annualized_yield"].fillna(0) +
        0.10 * out["openInterest"].fillna(0).rank(pct=True) +
        0.10 * out["volume"].fillna(0).rank(pct=True) +
        0.15 * out["strike"].rank(pct=True) +
        0.20 * out["iv_rank_proxy"].fillna(0) +
        0.15 * out["call_skew"].fillna(0) -
        0.05 * out["spread_pct"].fillna(0.5)
    )
    out["trade_framework"] = np.where(
        out["call_skew"].fillna(-999) >= max(MIN_CALL_SKEW_PREMIUM, 0.02),
        "SELL_CALL_ON_RICH_UPSIDE_SKEW",
        "SELL_CALL_STANDARD"
    )
    return out.sort_values(["score","annualized_yield"], ascending=[False, False]).reset_index(drop=True)

def build_new_cc_recommendations():
    rows = []
    for ticker, shares in STOCK_HOLDINGS.items():
        contracts = stock_contract_capacity(shares)
        if contracts <= 0:
            continue

        cost_basis = STOCK_COST_BASIS.get(ticker, np.nan)
        cc = scan_covered_calls(ticker, shares, cost_basis)
        if cc.empty:
            continue

        best = cc.iloc[0].copy()
        best["recommended_contracts"] = contracts
        best["expected_credit_total"] = float(best["credit_per_contract"]) * contracts
        rows.append(best)

    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows)
    return out.sort_values(["score","annualized_yield"], ascending=[False, False]).reset_index(drop=True)

# ============================================================
# 12) EXISTING SHORT OPTION MANAGEMENT
# ============================================================

def find_matching_contract_market(ticker, opt_type, expiry, strike):
    spot = safe_last_price(ticker)
    if pd.isna(spot):
        return None

    calls, puts = get_option_chain(ticker, expiry)
    df = puts if opt_type == "PUT" else calls
    if df.empty:
        return None

    norm_df = normalize_option_chain(df, ticker, expiry, opt_type, spot)
    if norm_df.empty:
        return None

    m = norm_df[np.isclose(norm_df["strike"].astype(float), float(strike))]
    if m.empty:
        return None

    return m.iloc[0].to_dict()

def manage_existing_short_positions_integrated(mc=None):
    mc = mc or get_market_context()
    short_df = summarize_short_option_positions()
    if short_df.empty:
        return pd.DataFrame()

    rows = []
    for _, r in short_df.iterrows():
        ticker = r["ticker"]
        opt_type = r["type"]
        expiry = r["expiry"]
        strike = float(r["strike"])
        contracts = int(r["contracts"])
        entry_credit = float(r["entry_credit"])
        dte = int(r["dte"])

        mkt = find_matching_contract_market(ticker, opt_type, expiry, strike)
        if mkt is None:
            continue

        current_mid = float(mkt.get("mid", np.nan))
        extrinsic = float(mkt.get("extrinsic", np.nan))
        spot = safe_last_price(ticker)
        current_iv, iv_rank_proxy = approx_iv_rank_proxy(ticker)

        profit_pct = np.nan
        if entry_credit > 0 and not pd.isna(current_mid):
            profit_pct = (entry_credit - current_mid) / entry_credit

        itm = (spot < strike) if opt_type == "PUT" else (spot > strike)

        action = "HOLD"
        reason = ""

        if not pd.isna(profit_pct) and profit_pct >= TARGET_PROFIT_CLOSE_PCT:
            action = "BUY_TO_CLOSE"
            reason = f"Reached {TARGET_PROFIT_CLOSE_PCT:.0%} profit target"

        elif dte <= ROLL_DTE_THRESHOLD:
            if itm:
                if not pd.isna(extrinsic) and extrinsic <= ITM_EXTRINSIC_CLOSE_THRESHOLD:
                    if opt_type == "PUT":
                        action = "LET_ASSIGN_OR_ROLL"
                        reason = "ITM put near expiry with low extrinsic"
                    else:
                        action = "LET_CALL_AWAY_OR_ROLL"
                        reason = "ITM call near expiry with low extrinsic"
                else:
                    action = "ROLL_DECISION"
                    reason = "Near expiry ITM; evaluate roll"
            else:
                action = "HOLD_TO_EXPIRY"
                reason = "OTM and near expiry"

        rr = get_ticker_risk_row(ticker, mc)
        if rr is not None and opt_type == "PUT" and action == "HOLD":
            close = safe_float(rr.get("close", np.nan))
            ma200 = safe_float(rr.get("ma200", np.nan))
            dd = safe_float(rr.get("dd", np.nan))

            if mc.get("crisis_on", False) or mc.get("regime") in ["RISK-OFF", "CRISIS"]:
                if ((not pd.isna(close) and not pd.isna(ma200) and close < ma200) or
                    (not pd.isna(dd) and dd < -0.20)):
                    action = "ROLL_DECISION"
                    reason = f"Macro defensive overlay | regime={mc.get('regime')}"

        rows.append({
            "ticker": ticker,
            "type": opt_type,
            "expiry": expiry,
            "dte": dte,
            "strike": strike,
            "contracts": contracts,
            "entry_credit": entry_credit,
            "current_mid": current_mid,
            "profit_pct": profit_pct,
            "spot": spot,
            "itm": itm,
            "extrinsic": extrinsic,
            "iv_current_proxy": current_iv,
            "iv_rank_proxy": iv_rank_proxy,
            "macro_regime": mc.get("regime", "NEUTRAL"),
            "macro_crisis_on": bool(mc.get("crisis_on", False)),
            "action": action,
            "reason": reason
        })

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values(["action","ticker","expiry"]).reset_index(drop=True)

# ============================================================
# 13) ROLL SCANNER
# ============================================================

def get_next_expiries(ticker, min_dte=21, max_dte=60):
    exps = get_option_expirations(ticker)
    if not exps:
        return []
    return [e for e in exps if min_dte <= days_to_expiry(e) <= max_dte]

def scan_roll_candidates_for_short_option(position, mc=None):
    mc = mc or get_market_context()

    ticker = position["ticker"]
    opt_type = position["type"]
    old_expiry = position["expiry"]
    old_strike = float(position["strike"])
    contracts = int(position["contracts"])

    current_contract = find_matching_contract_market(ticker, opt_type, old_expiry, old_strike)
    if current_contract is None:
        return pd.DataFrame()

    old_mid = safe_float(current_contract.get("mid", np.nan))
    if pd.isna(old_mid):
        return pd.DataFrame()

    spot = safe_last_price(ticker)
    if pd.isna(spot) or spot <= 0:
        return pd.DataFrame()

    current_iv, iv_rank_proxy = approx_iv_rank_proxy(ticker)
    next_exps = get_next_expiries(ticker, min_dte=ROLL_OUT_DTE_MIN, max_dte=ROLL_OUT_DTE_MAX)
    if not next_exps:
        return pd.DataFrame()

    rows = []
    regime = mc.get("regime", "NEUTRAL")

    for new_expiry in next_exps:
        calls, puts = get_option_chain(ticker, new_expiry)
        chain = puts if opt_type == "PUT" else calls
        if chain.empty:
            continue

        norm_chain = normalize_option_chain(chain, ticker, new_expiry, opt_type, spot)
        if norm_chain.empty:
            continue

        norm_chain = norm_chain[
            (norm_chain["openInterest"].fillna(0) >= MIN_OPEN_INTEREST) &
            (norm_chain["volume"].fillna(0) >= MIN_VOLUME) &
            (norm_chain["mid"].fillna(0) >= MIN_OPTION_PREMIUM) &
            (norm_chain["spread_pct"].fillna(999) <= MAX_BID_ASK_SPREAD_PCT)
        ]

        if norm_chain.empty:
            continue

        if opt_type == "PUT":
            cap = effective_put_delta_cap(ticker, mc)
            norm_chain = norm_chain[
                (norm_chain["delta_bs"].abs() >= ROLL_PUT_SAME_DELTA_MIN) &
                (norm_chain["delta_bs"].abs() <= min(ROLL_PUT_SAME_DELTA_MAX, cap))
            ]
            skew = get_put_skew_metrics(ticker, new_expiry, spot)
            if ENABLE_SKEW_FILTER and not pd.isna(skew["put_skew"]):
                norm_chain = norm_chain.copy()
                norm_chain["skew_value"] = skew["put_skew"]
        else:
            norm_chain = norm_chain[
                (norm_chain["delta_bs"].fillna(0) >= ROLL_CALL_SAME_DELTA_MIN) &
                (norm_chain["delta_bs"].fillna(0) <= ROLL_CALL_SAME_DELTA_MAX)
            ]
            skew = get_call_skew_metrics(ticker, new_expiry, spot)
            if ENABLE_SKEW_FILTER and not pd.isna(skew["call_skew"]):
                norm_chain = norm_chain.copy()
                norm_chain["skew_value"] = skew["call_skew"]

        if norm_chain.empty:
            continue

        if opt_type == "PUT" and PREFER_STRIKE_IMPROVEMENT:
            preferred = norm_chain[norm_chain["strike"] <= old_strike]
            if not preferred.empty:
                norm_chain = preferred
        elif opt_type == "CALL" and PREFER_STRIKE_IMPROVEMENT:
            preferred = norm_chain[norm_chain["strike"] >= old_strike]
            if not preferred.empty:
                norm_chain = preferred

        for _, cand in norm_chain.iterrows():
            new_mid = safe_float(cand["mid"])
            if pd.isna(new_mid):
                continue

            roll_credit = (new_mid - old_mid) * CONTRACT_MULTIPLIER * contracts
            new_dte = int(cand["dte"])
            strike_change = float(cand["strike"]) - old_strike

            if opt_type == "PUT":
                strike_improvement = old_strike - float(cand["strike"])
                delta_term = -abs(safe_float(cand["delta_bs"]))
            else:
                strike_improvement = float(cand["strike"]) - old_strike
                delta_term = -safe_float(cand["delta_bs"])

            score = (
                0.30 * int(roll_credit >= MIN_ROLL_CREDIT) +
                0.20 * strike_improvement +
                0.15 * safe_float(cand["annualized_yield"]) +
                0.15 * safe_float(iv_rank_proxy if not pd.isna(iv_rank_proxy) else 0) +
                0.10 * safe_float(cand.get("skew_value", 0)) +
                0.10 * delta_term
            )

            if regime in ["RISK-OFF", "CRISIS", "LATE-CYCLE"] and opt_type == "PUT":
                score += 0.10 * strike_improvement

            rows.append({
                "ticker": ticker,
                "type": opt_type,
                "old_expiry": old_expiry,
                "old_strike": old_strike,
                "old_mid": old_mid,
                "new_expiry": new_expiry,
                "new_strike": float(cand["strike"]),
                "new_mid": new_mid,
                "new_dte": new_dte,
                "delta_bs": safe_float(cand["delta_bs"]),
                "annualized_yield": safe_float(cand["annualized_yield"]),
                "iv_rank_proxy": iv_rank_proxy,
                "skew_value": safe_float(cand.get("skew_value", np.nan)),
                "strike_change": strike_change,
                "strike_improvement": strike_improvement,
                "roll_credit": roll_credit,
                "roll_for_credit": bool(roll_credit >= MIN_ROLL_CREDIT),
                "macro_regime": regime,
                "score": score
            })

    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows)
    out = out.sort_values(
        ["roll_for_credit", "score", "strike_improvement", "annualized_yield"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)
    return out

def build_roll_scanner_report(mc=None):
    mc = mc or get_market_context()
    short_df = summarize_short_option_positions()
    if short_df.empty:
        return pd.DataFrame(), {}

    summary_rows = []
    details = {}

    for _, r in short_df.iterrows():
        pos = {
            "ticker": r["ticker"],
            "type": r["type"],
            "expiry": r["expiry"],
            "strike": float(r["strike"]),
            "contracts": int(r["contracts"]),
            "entry_credit": float(r["entry_credit"])
        }

        roll_df = scan_roll_candidates_for_short_option(pos, mc=mc)
        key = f"{pos['ticker']}_{pos['type']}_{pos['expiry']}_{pos['strike']}"
        details[key] = roll_df

        if roll_df.empty:
            summary_rows.append({
                "ticker": pos["ticker"],
                "type": pos["type"],
                "old_expiry": pos["expiry"],
                "old_strike": pos["strike"],
                "best_new_expiry": np.nan,
                "best_new_strike": np.nan,
                "roll_credit": np.nan,
                "roll_for_credit": False,
                "comment": "No good roll candidate found"
            })
        else:
            best = roll_df.iloc[0]
            summary_rows.append({
                "ticker": pos["ticker"],
                "type": pos["type"],
                "old_expiry": pos["expiry"],
                "old_strike": pos["strike"],
                "best_new_expiry": best["new_expiry"],
                "best_new_strike": best["new_strike"],
                "roll_credit": best["roll_credit"],
                "roll_for_credit": best["roll_for_credit"],
                "comment": "Roll for credit" if best["roll_for_credit"] else "Roll requires debit"
            })

    summary = pd.DataFrame(summary_rows)
    return summary, details

# ============================================================
# 14) ASSIGNMENT SIMULATOR
# ============================================================

def simulate_put_assignment_and_next_cc(
    ticker,
    put_strike,
    put_entry_credit,
    contracts=1,
    assigned_shares=None
):
    contracts = int(contracts)
    if assigned_shares is None:
        assigned_shares = contracts * CONTRACT_MULTIPLIER

    put_strike = float(put_strike)
    put_entry_credit = float(put_entry_credit)

    assignment_cost_basis = put_strike - put_entry_credit
    total_put_credit = put_entry_credit * CONTRACT_MULTIPLIER * contracts
    stock_capital_at_risk = assignment_cost_basis * assigned_shares

    base = {
        "ticker": ticker,
        "put_strike": put_strike,
        "put_entry_credit": put_entry_credit,
        "assignment_cost_basis": assignment_cost_basis,
        "assigned_shares": assigned_shares,
        "total_put_credit": total_put_credit,
        "best_cc_found": False,
        "cc_expiry": np.nan,
        "cc_strike": np.nan,
        "cc_mid": np.nan,
        "cc_dte": np.nan,
        "total_cc_credit": np.nan,
        "capital_gain_if_called": np.nan,
        "total_wheel_profit_if_called": np.nan,
        "wheel_return_if_called": np.nan,
        "annualized_return_if_called": np.nan,
        "iv_rank_proxy": np.nan,
        "call_skew": np.nan,
    }

    cc_df = scan_covered_calls(
        ticker=ticker,
        shares_held=assigned_shares,
        cost_basis=assignment_cost_basis
    )

    if cc_df.empty:
        return base

    cc_df = cc_df[
        (cc_df["dte"] >= SIM_USE_CC_DTE_MIN) &
        (cc_df["dte"] <= SIM_USE_CC_DTE_MAX) &
        (cc_df["delta_bs"].fillna(0) >= SIM_TARGET_CC_DELTA_MIN) &
        (cc_df["delta_bs"].fillna(0) <= SIM_TARGET_CC_DELTA_MAX)
    ]

    if cc_df.empty:
        return base

    best_cc = cc_df.iloc[0]
    cc_strike = float(best_cc["strike"])
    cc_mid = float(best_cc["mid"])
    cc_dte = int(best_cc["dte"])
    total_cc_credit = cc_mid * CONTRACT_MULTIPLIER * contracts

    capital_gain_if_called = max(cc_strike - assignment_cost_basis, 0.0) * assigned_shares
    total_wheel_profit_if_called = total_put_credit + total_cc_credit + capital_gain_if_called

    wheel_return_if_called = (
        total_wheel_profit_if_called / stock_capital_at_risk
        if stock_capital_at_risk > 0 else np.nan
    )
    annualized_return_if_called = annualize_simple_return(wheel_return_if_called, cc_dte)

    base.update({
        "best_cc_found": True,
        "cc_expiry": best_cc["expiry"],
        "cc_strike": cc_strike,
        "cc_mid": cc_mid,
        "cc_dte": cc_dte,
        "total_cc_credit": total_cc_credit,
        "capital_gain_if_called": capital_gain_if_called,
        "total_wheel_profit_if_called": total_wheel_profit_if_called,
        "wheel_return_if_called": wheel_return_if_called,
        "annualized_return_if_called": annualized_return_if_called,
        "iv_rank_proxy": safe_float(best_cc.get("iv_rank_proxy", np.nan)),
        "call_skew": safe_float(best_cc.get("call_skew", np.nan)),
    })

    return base

def build_assignment_simulator_report():
    short_df = summarize_short_option_positions()
    if short_df.empty:
        return pd.DataFrame()

    puts = short_df[short_df["type"] == "PUT"].copy()
    if puts.empty:
        return pd.DataFrame()

    rows = []
    for _, r in puts.iterrows():
        sim = simulate_put_assignment_and_next_cc(
            ticker=r["ticker"],
            put_strike=float(r["strike"]),
            put_entry_credit=float(r["entry_credit"]),
            contracts=int(r["contracts"])
        )
        rows.append(sim)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["best_cc_found"] = out["best_cc_found"].fillna(False)

    return out.sort_values(
        ["best_cc_found", "annualized_return_if_called", "wheel_return_if_called"],
        ascending=[False, False, False],
        na_position="last"
    ).reset_index(drop=True)

# ============================================================
# 15) WHEEL STATE
# ============================================================

def wheel_state_per_ticker():
    holdings_df = current_stock_holdings_table()
    short_df = summarize_short_option_positions()

    rows = []
    for t in sorted(set(OPTION_UNIVERSE + list(STOCK_HOLDINGS.keys()))):
        shares = float(holdings_df.loc[t, "shares"]) if t in holdings_df.index else 0.0
        cc_capacity = stock_contract_capacity(shares)

        short_puts = 0
        short_calls = 0
        if not short_df.empty:
            sub = short_df[short_df["ticker"] == t]
            short_puts = int(sub[sub["type"] == "PUT"]["contracts"].sum()) if not sub.empty else 0
            short_calls = int(sub[sub["type"] == "CALL"]["contracts"].sum()) if not sub.empty else 0

        if cc_capacity > short_calls:
            stage = "COVERED_CALL_CANDIDATE"
        elif short_puts > 0:
            stage = "SHORT_PUT_OPEN"
        elif short_calls > 0:
            stage = "SHORT_CALL_OPEN"
        else:
            stage = "NEW_CSP_CANDIDATE"

        rows.append({
            "ticker": t,
            "shares": shares,
            "cc_capacity": cc_capacity,
            "short_put_contracts": short_puts,
            "short_call_contracts": short_calls,
            "stage": stage
        })

    return pd.DataFrame(rows).set_index("ticker").sort_index()

# ============================================================
# 16) EXECUTION PLAN
# ============================================================

def build_today_execution_plan_integrated(mc=None):
    mc = mc or get_market_context()

    holdings_df = current_stock_holdings_table()
    short_manage = manage_existing_short_positions_integrated(mc)
    wheel_state = wheel_state_per_ticker()
    new_csp = build_new_csp_recommendations_integrated(mc)
    new_cc = build_new_cc_recommendations()

    roll_summary, roll_details = build_roll_scanner_report(mc)
    assignment_report = build_assignment_simulator_report()

    plan_rows = []

    if not short_manage.empty:
        for _, r in short_manage.iterrows():
            plan_rows.append({
                "priority": 1,
                "ticker": r["ticker"],
                "action_type": r["action"],
                "instrument": r["type"],
                "expiry": r["expiry"],
                "strike": r["strike"],
                "contracts": r["contracts"],
                "est_credit_or_debit": r["current_mid"] * CONTRACT_MULTIPLIER if not pd.isna(r["current_mid"]) else np.nan,
                "note": r["reason"]
            })

    if not roll_summary.empty:
        for _, r in roll_summary.iterrows():
            if pd.isna(r["best_new_strike"]):
                continue
            plan_rows.append({
                "priority": 2,
                "ticker": r["ticker"],
                "action_type": "ROLL_CANDIDATE",
                "instrument": r["type"],
                "expiry": r["best_new_expiry"],
                "strike": r["best_new_strike"],
                "contracts": np.nan,
                "est_credit_or_debit": r["roll_credit"],
                "note": r["comment"]
            })

    if not new_cc.empty:
        for _, r in new_cc.head(10).iterrows():
            plan_rows.append({
                "priority": 3,
                "ticker": r["ticker"],
                "action_type": "SELL_TO_OPEN",
                "instrument": "CALL",
                "expiry": r["expiry"],
                "strike": r["strike"],
                "contracts": int(r["recommended_contracts"]),
                "est_credit_or_debit": float(r["expected_credit_total"]),
                "note": (
                    f"CC | delta={r['delta_bs']:.2f} | ann_yield={r['annualized_yield']:.2%} | "
                    f"IVR={safe_float(r.get('iv_rank_proxy', np.nan)):.2f} | call_skew={safe_float(r.get('call_skew', np.nan)):.4f}"
                )
            })

    if not new_csp.empty and not (mc.get("crisis_on", False) or mc.get("regime") == "CRISIS"):
        for _, r in new_csp.head(10).iterrows():
            plan_rows.append({
                "priority": 4,
                "ticker": r["ticker"],
                "action_type": "SELL_TO_OPEN",
                "instrument": "PUT",
                "expiry": r["expiry"],
                "strike": r["strike"],
                "contracts": int(r["recommended_contracts"]),
                "est_credit_or_debit": float(r["expected_credit_total"]),
                "note": (
                    f"CSP | delta={r['delta_bs']:.2f} | ann_yield={r['annualized_yield']:.2%} | "
                    f"IVR={safe_float(r.get('iv_rank_proxy', np.nan)):.2f} | put_skew={safe_float(r.get('put_skew', np.nan)):.4f} | "
                    f"cash_req={r['cash_required']:.0f} | regime={mc.get('regime')}"
                )
            })

    plan = pd.DataFrame(plan_rows)
    if not plan.empty:
        plan = plan.sort_values(["priority","ticker"]).reset_index(drop=True)

    return {
        "holdings_df": holdings_df,
        "short_manage": short_manage,
        "wheel_state": wheel_state,
        "new_csp": new_csp,
        "new_cc": new_cc,
        "roll_summary": roll_summary,
        "roll_details": roll_details,
        "assignment_report": assignment_report,
        "plan": plan,
        "market_context": mc,
    }

# ============================================================
# 17) CSV EXPORT
# ============================================================

def export_results_to_csv_integrated(res, output_dir="wheel_outputs"):
    os.makedirs(output_dir, exist_ok=True)

    snap = wheel_capital_snapshot_integrated(res.get("market_context"))
    pd.Series(snap).to_frame("value").to_csv(f"{output_dir}/capital_snapshot.csv")

    mc = res.get("market_context", {})
    if mc:
        try:
            pd.Series({
                "regime": mc.get("regime"),
                "crisis_on": mc.get("crisis_on"),
                "L_final": safe_float(mc.get("L_final", np.nan)),
                "max_gross": safe_float(mc.get("limits", {}).get("max_gross", np.nan)),
                "cash_floor": safe_float(mc.get("limits", {}).get("cash_floor", np.nan)),
            }).to_frame("value").to_csv(f"{output_dir}/macro_context_summary.csv")
        except Exception:
            pass

        if mc.get("stock_signals") is not None and not mc["stock_signals"].empty:
            mc["stock_signals"].to_csv(f"{output_dir}/stock_signals.csv")

        tw = mc.get("target_weights", pd.Series(dtype=float))
        if tw is not None and len(tw) > 0:
            tw.to_frame("target_weight").to_csv(f"{output_dir}/target_weights.csv")

        if mc.get("risk_indicators") is not None and not mc["risk_indicators"].empty:
            mc["risk_indicators"].to_csv(f"{output_dir}/risk_indicators.csv")

        if mc.get("macro") is not None:
            if mc["macro"].get("factors") is not None and not mc["macro"]["factors"].empty:
                mc["macro"]["factors"].to_csv(f"{output_dir}/macro_factors.csv")
            if mc["macro"].get("Z") is not None and not mc["macro"]["Z"].empty:
                mc["macro"]["Z"].to_csv(f"{output_dir}/macro_zscores.csv")

    if res.get("wheel_state") is not None and not res["wheel_state"].empty:
        res["wheel_state"].reset_index().to_csv(f"{output_dir}/wheel_state.csv", index=False)

    if res.get("holdings_df") is not None and not res["holdings_df"].empty:
        res["holdings_df"].reset_index().to_csv(f"{output_dir}/stock_holdings.csv", index=False)

    if res.get("short_manage") is not None and not res["short_manage"].empty:
        res["short_manage"].to_csv(f"{output_dir}/manage_short_options.csv", index=False)

    if res.get("roll_summary") is not None and not res["roll_summary"].empty:
        res["roll_summary"].to_csv(f"{output_dir}/roll_summary.csv", index=False)

    if res.get("assignment_report") is not None and not res["assignment_report"].empty:
        res["assignment_report"].to_csv(f"{output_dir}/assignment_report.csv", index=False)

    if res.get("new_csp") is not None and not res["new_csp"].empty:
        res["new_csp"].to_csv(f"{output_dir}/new_csp_candidates.csv", index=False)

    if res.get("new_cc") is not None and not res["new_cc"].empty:
        res["new_cc"].to_csv(f"{output_dir}/new_covered_call_candidates.csv", index=False)

    if res.get("plan") is not None and not res["plan"].empty:
        res["plan"].to_csv(f"{output_dir}/today_execution_plan.csv", index=False)

    roll_details = res.get("roll_details", {})
    if isinstance(roll_details, dict):
        for k, v in roll_details.items():
            if v is not None and not v.empty:
                safe_name = str(k).replace("/", "_").replace(":", "_")
                v.to_csv(f"{output_dir}/roll_detail_{safe_name}.csv", index=False)

    print(f"CSV exported to: {output_dir}")

# ============================================================
# 18) RUNNER
# ============================================================

def print_market_context_summary(mc):
    print("=== MARKET CONTEXT ===")
    print("regime:", mc.get("regime"))
    print("crisis_on:", mc.get("crisis_on"))
    print("L_final:", safe_float(mc.get("L_final", np.nan)))
    print("limits:", mc.get("limits", {}))

    pr = mc.get("portfolio_risk", None)
    if pr:
        print("portfolio_risk:", pr)

    macro = mc.get("macro", None)
    if macro:
        print("crisis_flags:", macro.get("crisis_flags", {}))
        print("latest_macro:", macro.get("latest", {}))

def run_options_daily_integrated(show_roll_details=False, refresh_context_first=False):
    mc = refresh_market_context() if refresh_context_first else get_market_context()

    snap = wheel_capital_snapshot_integrated(mc)

    print("=== CAPITAL SNAPSHOT ===")
    print(pd.Series(snap))
    print()
    print_market_context_summary(mc)

    res = build_today_execution_plan_integrated(mc)

    print("\n=== WHEEL STATE ===")
    safe_display(res["wheel_state"])

    print("\n=== STOCK HOLDINGS (for CC only) ===")
    safe_display(
        res["holdings_df"],
        round_cols=["price","market_value","cost_basis"],
        pct_cols=["unreal_pnl_pct"]
    )

    print("\n=== MANAGE EXISTING SHORT OPTIONS ===")
    if res["short_manage"].empty:
        print("No existing short option positions.")
    else:
        safe_display(
            res["short_manage"],
            round_cols=["strike","entry_credit","current_mid","spot","extrinsic","iv_rank_proxy"],
            pct_cols=["profit_pct"]
        )

    print("\n=== ROLL SCANNER SUMMARY ===")
    if res["roll_summary"].empty:
        print("No roll candidates.")
    else:
        safe_display(
            res["roll_summary"],
            round_cols=["old_strike","best_new_strike","roll_credit"]
        )

    if show_roll_details and res["roll_details"]:
        print("\n=== ROLL SCANNER DETAILS ===")
        for k, v in res["roll_details"].items():
            print(f"\n--- {k} ---")
            if v is None or v.empty:
                print("No details.")
            else:
                safe_display(
                    v.head(10),
                    round_cols=[
                        "old_strike","old_mid","new_strike","new_mid","delta_bs",
                        "annualized_yield","strike_change","strike_improvement",
                        "roll_credit","score","iv_rank_proxy","skew_value"
                    ]
                )

    print("\n=== ASSIGNMENT SIMULATOR ===")
    if res["assignment_report"].empty:
        print("No short put positions to simulate assignment.")
    else:
        safe_display(
            res["assignment_report"],
            round_cols=[
                "put_strike","put_entry_credit","assignment_cost_basis",
                "total_put_credit","cc_strike","cc_mid","total_cc_credit",
                "capital_gain_if_called","total_wheel_profit_if_called","iv_rank_proxy","call_skew"
            ],
            pct_cols=["wheel_return_if_called","annualized_return_if_called"]
        )

    print("\n=== NEW CSP CANDIDATES ===")
    if res["new_csp"].empty:
        print("No new CSP candidates.")
    else:
        show_cols = [
            "ticker","expiry","strike","dte","mid","delta_bs","annualized_yield",
            "openInterest","volume","spread_pct","iv_rank_proxy","put_skew",
            "recommended_contracts","cash_required","expected_credit_total",
            "macro_regime","macro_L_final","semi_hot","trade_framework","score"
        ]
        show_cols = [c for c in show_cols if c in res["new_csp"].columns]

        safe_display(
            res["new_csp"][show_cols].head(DISPLAY_TOP_CSP),
            round_cols=[
                "strike","mid","delta_bs","annualized_yield","spread_pct","iv_rank_proxy",
                "put_skew","cash_required","expected_credit_total","macro_L_final","score"
            ]
        )

    print("\n=== NEW COVERED CALL CANDIDATES ===")
    if res["new_cc"].empty:
        print("No new covered call candidates.")
    else:
        safe_display(
            res["new_cc"][[
                "ticker","expiry","strike","dte","mid","delta_bs","annualized_yield",
                "openInterest","volume","spread_pct","cost_basis","iv_rank_proxy","call_skew",
                "recommended_contracts","expected_credit_total","trade_framework","score"
            ]].head(DISPLAY_TOP_CC),
            round_cols=[
                "strike","mid","delta_bs","annualized_yield","spread_pct","cost_basis",
                "iv_rank_proxy","call_skew","expected_credit_total","score"
            ]
        )

    print("\n=== TODAY EXECUTION PLAN (MANUAL ONLY) ===")
    if res["plan"].empty:
        print("No action today.")
    else:
        safe_display(
            res["plan"].head(DISPLAY_TOP_PLAN),
            round_cols=["strike","est_credit_or_debit"]
        )

    return res

# ============================================================
# 19) RUN NOW
# ============================================================

try:
    res = run_options_daily_integrated(show_roll_details=False, refresh_context_first=False)
    export_results_to_csv_integrated(res, output_dir="wheel_outputs")
except Exception as e:
    print("RUN FAILED:", e)
    traceback.print_exc()

=== CAPITAL SNAPSHOT ===
stock_mv                          0.0
short_put_cash_obligation     16800.0
total_deployed                16800.0
max_total_deploy             8693.615
available_deploy_room             0.0
cash_buffer_required         8693.615
free_cash_now                  8582.9
macro_regime                 RISK-OFF
macro_crisis_on                  True
L_final                           0.0
dtype: object

=== MARKET CONTEXT ===
regime: RISK-OFF
crisis_on: True
L_final: 0.0
limits: {'max_gross': 0.35, 'max_net': 0.2, 'max_leverage': 0.0, 'cash_floor': 0.35, 'notes': ['CRISIS：槓桿=0，現金≥35%']}
crisis_flags: {'VIX > threshold': np.True_, 'CreditStress > threshold': True, 'VolScore > threshold': True, 'Dislocation > thr': False}
latest_macro: {'Liquidity': 3.0620653412625516, 'Credit': 1.118249828797524, 'Volatility': 3.2140123038251818, 'Growth': 2.651362122047963, 'Rate': 1.7269524568763241, 'Geo': 3.2935695710976725, 'Dislocation': -0.3365739451829516, 'TotalScore': 2.4558345254

,shares,cc_capacity,short_put_contracts,short_call_contracts,stage
ticker,,,,,
AAPL,0.0,0,0,0,NEW_CSP_CANDIDATE
AMAT,0.0,0,0,0,NEW_CSP_CANDIDATE
AMD,0.0,0,0,0,NEW_CSP_CANDIDATE
AMZN,0.0,0,0,0,NEW_CSP_CANDIDATE
ASML,0.0,0,0,0,NEW_CSP_CANDIDATE
GOOGL,0.0,0,0,0,NEW_CSP_CANDIDATE
INTC,0.0,0,4,0,SHORT_PUT_OPEN
META,0.0,0,0,0,NEW_CSP_CANDIDATE
MSFT,0.0,0,0,0,NEW_CSP_CANDIDATE



=== STOCK HOLDINGS (for CC only) ===
No data.

=== MANAGE EXISTING SHORT OPTIONS ===


,ticker,type,expiry,dte,strike,contracts,entry_credit,current_mid,profit_pct,spot,itm,extrinsic,iv_current_proxy,iv_rank_proxy,macro_regime,macro_crisis_on,action,reason
0,INTC,PUT,2026-04-02,10,44.0,1,1.50,2.115,-41.0,43.87,True,1.985,0.629154,0.6753,RISK-OFF,True,HOLD,
1,INTC,PUT,2026-04-02,10,40.0,1,0.50,0.730,-46.0,43.87,False,0.730,0.629154,0.6753,RISK-OFF,True,HOLD,
2,INTC,PUT,2026-04-02,10,42.0,2,0.92,1.265,-37.5,43.87,False,1.265,0.629154,0.6753,RISK-OFF,True,HOLD,



=== ROLL SCANNER SUMMARY ===


...


=== ASSIGNMENT SIMULATOR ===


,ticker,put_strike,put_entry_credit,assignment_cost_basis,assigned_shares,total_put_credit,best_cc_found,cc_expiry,cc_strike,cc_mid,cc_dte,total_cc_credit,capital_gain_if_called,total_wheel_profit_if_called,wheel_return_if_called,annualized_return_if_called,iv_rank_proxy,call_skew
0,INTC,44.0,1.50,42.50,100,150.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,INTC,40.0,0.50,39.50,100,50.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,INTC,42.0,0.92,41.08,200,184.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== NEW CSP CANDIDATES ===
No new CSP candidates.

=== NEW COVERED CALL CANDIDATES ===
No new covered call candidates.

=== TODAY EXECUTION PLAN (MANUAL ONLY) ===


,priority,ticker,action_type,instrument,expiry,strike,contracts,est_credit_or_debit,note
0,1,INTC,HOLD,PUT,2026-04-02,44.0,1,211.5,
1,1,INTC,HOLD,PUT,2026-04-02,40.0,1,73.0,
2,1,INTC,HOLD,PUT,2026-04-02,42.0,2,126.5,


CSV exported to: wheel_outputs
